# 🦾 Chopstick Crane — Interactive Notebook

This notebook walks you through every piece of the problem — from first principles maths to running the full simulation. Use the sliders and interactive widgets to build intuition.

---

## Sections
1. [Forward Kinematics — Interactive](#s1)  
2. [Jacobian Explorer](#s2)  
3. [IK Demo — Click Your Target](#s3)  
4. [Board Spring Dynamics (LTI System)](#s4)  
5. [Control Theory Deep-Dive](#s5)  
6. [Run Full Simulation & Replay](#s6)  

---

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, FloatSlider, IntSlider, Layout

# Import our modules
import sys, os
sys.path.insert(0, os.getcwd())

from kinematics import (
    forward_kinematics, jacobian, numerical_jacobian,
    all_joint_positions, dls_ik_step, clamp_joints,
    target_curve, target_with_offset, board_normal, board_tangent,
    board_to_world, ik_init_guess,
    L1, L2, L3, BOARD_HINGE_POS, SWEEP_OFFSET, SWEEP_AMPLITUDE,
    F_MIN, F_MAX, F_TARGET, BOARD_K, BOARD_B, BOARD_INERTIA,
)

# Dark matplotlib style
plt.rcParams.update({
    'figure.facecolor':  '#12121e',
    'axes.facecolor':    '#1a1a2e',
    'axes.edgecolor':    '#3a3a5c',
    'axes.labelcolor':   '#c8c8d8',
    'axes.titlecolor':   '#e8e8f0',
    'axes.grid':          True,
    'grid.color':        '#2a2a4a',
    'xtick.color':       '#a0a0c0',
    'ytick.color':       '#a0a0c0',
    'text.color':        '#c8c8d8',
    'lines.linewidth':    2.0,
    'font.size':         11,
})
print('✅ Setup complete!')

✅ Setup complete!


---
<a id='s1'></a>
## Section 1 — Forward Kinematics (Interactive)

### The Maths

The arm lives in the **x-z plane** (gravity in −z). All joints rotate about the y-axis.

Let cumulative angles be: $\alpha_1 = \theta_1$, $\alpha_2 = \theta_1+\theta_2$, $\alpha_3 = \theta_1+\theta_2+\theta_3$

$$
\begin{bmatrix} p_x \\ p_z \end{bmatrix} =
L_1 \begin{bmatrix} \cos\alpha_1 \\ \sin\alpha_1 \end{bmatrix} +
L_2 \begin{bmatrix} \cos\alpha_2 \\ \sin\alpha_2 \end{bmatrix} +
L_3 \begin{bmatrix} \cos\alpha_3 \\ \sin\alpha_3 \end{bmatrix}
$$

**Drag the sliders** to see the arm move:

In [2]:
def draw_arm(theta1=30, theta2=-40, theta3=20, show_board=True, show_reach=False):
    """Draw the 3-link arm for given joint angles (degrees)."""
    theta = np.radians([theta1, theta2, theta3])
    pts   = all_joint_positions(theta)
    tip   = forward_kinematics(theta)

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.set_title('Forward Kinematics Explorer', fontsize=14)
    ax.set_xlim(-0.8, 1.0);  ax.set_ylim(-0.7, 0.8)
    ax.set_aspect('equal');  ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')

    # Reachable workspace circle (rough)
    if show_reach:
        r_max = L1 + L2 + L3
        circle = plt.Circle((0, 0), r_max, fill=False, color='#334466', ls='--', lw=1)
        ax.add_patch(circle)
        ax.text(r_max * 0.72, r_max * 0.72, 'Max reach', color='#334466', fontsize=8)

    # Draw board
    if show_board:
        board_len = 0.30
        hx, hz = BOARD_HINGE_POS
        ax.plot([hx, hx + board_len], [hz, hz],
                color='#aa8855', lw=6, solid_capstyle='round', alpha=0.7, label='Board (φ=0)')
        ax.scatter(hx, hz, color='#ffdd44', s=80, zorder=5, label='Hinge A')
        # Target curve on board
        s_vals = np.linspace(0, 1, 100)
        curve  = np.array([target_curve(s, 0.0) for s in s_vals])
        ax.plot(curve[:, 0], curve[:, 1], color='#33dd88', lw=2, ls='--', label='Target curve')

    # Draw links
    colours = ['#ff6688', '#66aaff', '#88ff99']
    labels  = ['Link 1 (L1=0.30m)', 'Link 2 (L2=0.25m)', 'Link 3 (L3=0.15m)']
    for i in range(3):
        ax.plot([pts[i][0], pts[i+1][0]], [pts[i][1], pts[i+1][1]],
                color=colours[i], lw=5, solid_capstyle='round', label=labels[i])

    # Draw joints
    for i, (x, z) in enumerate(pts):
        ax.scatter(x, z, color='#ffdd44', s=60 - i*10, zorder=5)

    # Pen tip
    ax.scatter(*tip, color='#ff3333', s=100, zorder=6, label=f'Pen tip  ({tip[0]:.3f}, {tip[1]:.3f}) m')

    # Origin
    ax.scatter(0, 0, color='white', s=80, marker='s', zorder=7)
    ax.text(0.02, 0.03, 'Base', color='white', fontsize=9)

    # Cumulative angles
    alpha = np.degrees(np.cumsum(theta))
    ax.text(0.02, -0.62,
            f'α₁={alpha[0]:.1f}°   α₂={alpha[1]:.1f}°   α₃={alpha[2]:.1f}°',
            color='#aaaacc', fontsize=9)

    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()


slider_kw = dict(min=-180, max=180, step=1, continuous_update=True,
                 layout=Layout(width='450px'))

interact(
    draw_arm,
    theta1 = FloatSlider(value=30,  description='θ₁ (shoulder) [°]', **slider_kw),
    theta2 = FloatSlider(value=-40, description='θ₂ (elbow) [°]',    **slider_kw),
    theta3 = FloatSlider(value=20,  description='θ₃ (wrist) [°]',    **slider_kw),
    show_board = widgets.Checkbox(value=True,  description='Show board'),
    show_reach = widgets.Checkbox(value=False, description='Show workspace'),
);

interactive(children=(FloatSlider(value=30.0, description='θ₁ (shoulder) [°]', layout=Layout(width='450px'), m…

---
<a id='s2'></a>
## Section 2 — Jacobian Explorer

### What is the Jacobian?

The Jacobian $J(\theta)$ maps **joint velocity** → **end-effector velocity**:

$$\dot{p} = J(\theta)\,\dot{\theta}, \quad J \in \mathbb{R}^{2 \times 3}$$

For our arm:

$$J = \begin{bmatrix}
-L_1 s_1 - L_2 s_{12} - L_3 s_{123} & -L_2 s_{12} - L_3 s_{123} & -L_3 s_{123} \\
 L_1 c_1 + L_2 c_{12} + L_3 c_{123} &  L_2 c_{12} + L_3 c_{123} &  L_3 c_{123}
\end{bmatrix}$$

where $c_{ij} = \cos(\theta_i+\theta_j)$, etc.

The **columns** of J are the velocity the tip gets from spinning each joint alone.  
The **condition number** of J tells us how close we are to a **singularity** (where the arm loses mobility).

In [3]:
def explore_jacobian(theta1=30, theta2=-40, theta3=20):
    theta = np.radians([theta1, theta2, theta3])
    J     = jacobian(theta)
    J_num = numerical_jacobian(theta)
    pts   = all_joint_positions(theta)
    tip   = forward_kinematics(theta)

    # Condition number and manipulability
    svd   = np.linalg.svd(J, compute_uv=False)
    cond  = svd[0] / (svd[-1] + 1e-10)
    manip = np.sqrt(np.linalg.det(J @ J.T))

    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    # Left: arm + Jacobian columns (velocity ellipsoid)
    ax = axes[0]
    ax.set_title('Arm + Jacobian Column Vectors', fontsize=12)
    ax.set_xlim(-0.8, 1.0);  ax.set_ylim(-0.7, 0.8)
    ax.set_aspect('equal');  ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')

    colours = ['#ff6688', '#66aaff', '#88ff99']
    for i in range(3):
        ax.plot([pts[i][0], pts[i+1][0]], [pts[i][1], pts[i+1][1]],
                color=colours[i], lw=5, solid_capstyle='round')
    ax.scatter(*tip, color='#ff3333', s=100, zorder=6)
    ax.scatter(0, 0, color='white', s=80, marker='s', zorder=7)

    # Draw Jacobian columns (velocity contributions per joint)
    arrow_scale = 0.6
    col_labels  = ['∂p/∂θ₁', '∂p/∂θ₂', '∂p/∂θ₃']
    for i in range(3):
        col = J[:, i] * arrow_scale
        ax.annotate('', xy=(tip[0] + col[0], tip[1] + col[1]), xytext=(tip[0], tip[1]),
                    arrowprops=dict(arrowstyle='->', color=colours[i], lw=2.5))
        ax.text(tip[0] + col[0] + 0.01, tip[1] + col[1] + 0.01,
                col_labels[i], color=colours[i], fontsize=9)

    # Draw velocity ellipse (manipulability)
    U, S, _ = np.linalg.svd(J)
    theta_e  = np.linspace(0, 2*np.pi, 100)
    ellipse  = U @ np.diag(S * 0.15) @ np.array([np.cos(theta_e), np.sin(theta_e)])
    ax.plot(tip[0] + ellipse[0], tip[1] + ellipse[1],
            color='#ffdd44', lw=1.5, ls=':', alpha=0.8, label='Velocity ellipse')
    ax.legend(fontsize=9)

    # Right: J matrix heatmap + stats
    ax2 = axes[1]
    ax2.set_title('Jacobian Matrix Values', fontsize=12)

    im = ax2.imshow(J, aspect='auto', cmap='RdBu', vmin=-1.0, vmax=1.0)
    ax2.set_xticks([0, 1, 2]);  ax2.set_xticklabels(['θ₁', 'θ₂', 'θ₃'])
    ax2.set_yticks([0, 1]);     ax2.set_yticklabels(['pₓ', 'pz'])
    plt.colorbar(im, ax=ax2, fraction=0.046)

    for i in range(2):
        for j in range(3):
            ax2.text(j, i, f'{J[i,j]:.3f}', ha='center', va='center',
                     color='white', fontsize=10, fontweight='bold')

    ax2.text(0.02, -0.22, f'Singular values: {svd.round(3)}', transform=ax2.transAxes,
             color='#aaaacc', fontsize=9)
    ax2.text(0.02, -0.30, f'Condition number: {cond:.2f}   (> 20 ⇒ near singular!)',
             transform=ax2.transAxes, color='#ffaa44' if cond > 20 else '#44ff99', fontsize=9)
    ax2.text(0.02, -0.38, f'Manipulability:  {manip:.4f}   (0 = singular)',
             transform=ax2.transAxes, color='#aaaacc', fontsize=9)
    ax2.text(0.02, -0.46, f'|J_num − J_ana| max: {np.abs(J_num-J).max():.2e}   (should be ~0)',
             transform=ax2.transAxes, color='#88ddff', fontsize=9)

    plt.tight_layout()
    plt.show()


interact(
    explore_jacobian,
    theta1 = FloatSlider(value=30,  min=-180, max=180, step=1, description='θ₁ [°]'),
    theta2 = FloatSlider(value=-40, min=-150, max=150, step=1, description='θ₂ [°]'),
    theta3 = FloatSlider(value=20,  min=-180, max=180, step=1, description='θ₃ [°]'),
);

interactive(children=(FloatSlider(value=30.0, description='θ₁ [°]', max=180.0, min=-180.0, step=1.0), FloatSli…

---
<a id='s3'></a>
## Section 3 — IK Demo: Iterative Solver

### Damped Least-Squares IK (DLS)

Given desired pen-tip position $p^*$, find $\theta$ such that $FK(\theta) = p^*$.

**Why not just invert?** Because $J$ is $2\times3$ (more unknowns than equations), so there are infinitely many solutions. We pick the one that:
1. Gets the tip to $p^*$ (primary task)
2. Keeps the arm close to its home config (secondary task via null-space)

**DLS update rule:**
$$\Delta\theta = \underbrace{J^\top(JJ^\top + \lambda^2 I)^{-1}}_{J^\dagger_{\text{DLS}}} \Delta p + \underbrace{(I - J^\dagger_{\text{DLS}} J)}_{\text{null-space projector}} \Delta\theta_0$$

- $\lambda$ = damping → larger $\lambda$ = more robust near singularities (but slower)
- $(I - J^\dagger J)$ projects any vector into the null-space (zero task-space effect)
- $\Delta\theta_0 = $ secondary task gradient (e.g., home config bias)

In [4]:
# ── Interactive IK solver ──────────────────────────────────────────────────

def run_ik_demo(target_x=0.40, target_z=-0.15, lambda_damp=0.02, n_iter=200,
                use_null_space=True, show_trace=True):
    """Solve IK iteratively and show convergence."""
    target    = np.array([target_x, target_z])
    theta     = ik_init_guess(target)             # closed-form seed
    theta_home= theta.copy()

    errors, thetas = [np.linalg.norm(forward_kinematics(theta) - target)], [theta.copy()]

    for _ in range(n_iter):
        dp     = target - forward_kinematics(theta)
        dtheta = dls_ik_step(
            theta, dp, lambda_damp=lambda_damp,
            theta_preferred=theta_home if use_null_space else None,
            null_gain=0.3,
        )
        theta  = clamp_joints(theta + dtheta)
        err    = np.linalg.norm(forward_kinematics(theta) - target)
        errors.append(err)
        thetas.append(theta.copy())
        if err < 1e-5:
            break

    tip_final = forward_kinematics(theta)

    # ── Plots ──
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    # Left: arm in final position
    ax = axes[0]
    ax.set_title(f'IK Solution  (iters={len(errors)-1}, err={errors[-1]*1000:.2f}mm)', fontsize=12)
    ax.set_xlim(-0.8, 1.0);  ax.set_ylim(-0.7, 0.8)
    ax.set_aspect('equal');  ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')

    # Target
    ax.scatter(*target,    color='#44ff99', s=150, marker='*', zorder=7, label=f'Target ({target_x:.2f}, {target_z:.2f})')
    ax.scatter(*tip_final, color='#ff3333', s=80,  zorder=6, label=f'Tip   ({tip_final[0]:.3f}, {tip_final[1]:.3f})')

    # Arm trace (every 10th iteration)
    if show_trace:
        for i in range(0, len(thetas)-1, max(1, len(thetas)//8)):
            pts_i = all_joint_positions(thetas[i])
            for j in range(3):
                ax.plot([pts_i[j][0], pts_i[j+1][0]], [pts_i[j][1], pts_i[j+1][1]],
                        color='#334466', lw=2, alpha=0.3)

    # Final arm
    pts = all_joint_positions(theta)
    colours = ['#ff6688', '#66aaff', '#88ff99']
    for i in range(3):
        ax.plot([pts[i][0], pts[i+1][0]], [pts[i][1], pts[i+1][1]],
                color=colours[i], lw=5, solid_capstyle='round')
    ax.scatter(0, 0, color='white', s=80, marker='s')
    ax.legend(fontsize=9)

    # Right: convergence plot
    ax2 = axes[1]
    ax2.set_title('Convergence', fontsize=12)
    ax2.set_xlabel('Iteration');  ax2.set_ylabel('Error [mm]')
    ax2.plot(np.array(errors) * 1000, color='#ff9944', lw=2)
    ax2.axhline(0.01, color='#44ff99', ls='--', lw=1, label='0.01 mm threshold')
    ax2.set_yscale('log');  ax2.legend(fontsize=9)

    # Joint angle display
    ax2.text(0.98, 0.95,
             f'θ₁ = {np.degrees(theta[0]):.1f}°\nθ₂ = {np.degrees(theta[1]):.1f}°\nθ₃ = {np.degrees(theta[2]):.1f}°',
             transform=ax2.transAxes, ha='right', va='top',
             color='#c8c8d8', fontsize=10, family='monospace',
             bbox=dict(boxstyle='round', fc='#1a1a2e', ec='#3a3a5c'))

    plt.tight_layout()
    plt.show()


interact(
    run_ik_demo,
    target_x    = FloatSlider(value=0.40, min=-0.7, max=0.9, step=0.01, description='Target x [m]'),
    target_z    = FloatSlider(value=-0.15, min=-0.6, max=0.6, step=0.01, description='Target z [m]'),
    lambda_damp = FloatSlider(value=0.02, min=0.001, max=0.20, step=0.001,
                              description='λ (damping)', readout_format='.3f'),
    n_iter      = IntSlider(value=200, min=5, max=500, step=5, description='Max iters'),
    use_null_space = widgets.Checkbox(value=True,  description='Use null-space'),
    show_trace     = widgets.Checkbox(value=True,  description='Show arm trace'),
);

interactive(children=(FloatSlider(value=0.4, description='Target x [m]', max=0.9, min=-0.7, step=0.01), FloatS…

---
<a id='s4'></a>
## Section 4 — Board Spring Dynamics (LTI System)

### The board is a 2nd-order system

The board hinge obeys:

$$I\ddot{\varphi} + b\dot{\varphi} + k\varphi = \tau_{\text{contact}}$$

Transfer function (Laplace domain):

$$\frac{\varphi(s)}{\tau(s)} = \frac{1}{Is^2 + bs + k}$$

This is a **damped harmonic oscillator** (spring-mass-damper). Parameters:
- Natural frequency: $\omega_n = \sqrt{k/I}$
- Damping ratio: $\zeta = b / (2\sqrt{kI})$
- Underdamped if $\zeta < 1$ (oscillates), overdamped if $\zeta > 1$

In [5]:
from scipy import signal

def plot_board_dynamics(k=3.0, b=0.3, I=0.02, tau_step=0.5):
    """Step response and Bode plot for the board spring system."""
    wn   = np.sqrt(k / I)
    zeta = b / (2 * np.sqrt(k * I))
    print(f'ωₙ = {wn:.2f} rad/s   ζ = {zeta:.3f}   '
          f'({"underdamped" if zeta < 1 else "overdamped" if zeta > 1 else "critically damped"})')

    sys = signal.TransferFunction([1], [I, b, k])

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Step response
    t_end = max(5 * (1/wn), 3)
    t, y  = signal.step(sys, T=np.linspace(0, t_end, 1000))
    y_deg = np.degrees(y * tau_step)   # scale by torque input

    ax = axes[0]
    ax.set_title(f'Step Response  (τ_step={tau_step} N·m)', fontsize=12)
    ax.set_xlabel('Time [s]');  ax.set_ylabel('Board tilt φ [deg]')
    ax.plot(t, y_deg, color='#66aaff', lw=2, label='φ(t)')
    ax.axhline(np.degrees(tau_step/k), color='#44ff99', ls='--', lw=1.5,
               label=f'Steady state = {np.degrees(tau_step/k):.2f}°')
    if zeta < 1:
        # Mark 2% settling time
        settle = (4.0 / (zeta * wn))
        ax.axvline(settle, color='#ff9944', ls=':', lw=1.5, label=f'Settle ≈ {settle:.2f}s')
    ax.legend(fontsize=9)

    # Bode plot
    w, H = signal.freqs([1], [I, b, k], worN=np.logspace(-1, 2, 300))
    ax2 = axes[1]
    ax2.set_title('Frequency Response (Bode magnitude)', fontsize=12)
    ax2.set_xlabel('Frequency [rad/s]');  ax2.set_ylabel('|H(jω)| [1/N·m]')
    ax2.loglog(w, np.abs(H), color='#ff9944', lw=2)
    ax2.axvline(wn, color='#44ff99', ls='--', lw=1.5, label=f'ωₙ = {wn:.2f} rad/s')
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.show()


interact(
    plot_board_dynamics,
    k      = FloatSlider(value=BOARD_K, min=0.5, max=20.0, step=0.5, description='k [N·m/rad]'),
    b      = FloatSlider(value=BOARD_B, min=0.0, max=5.0,  step=0.05, description='b [N·m·s/rad]'),
    I      = FloatSlider(value=BOARD_INERTIA, min=0.005, max=0.2, step=0.005, description='I [kg·m²]'),
    tau_step = FloatSlider(value=0.5, min=0.05, max=3.0, step=0.05, description='τ_step [N·m]'),
);

interactive(children=(FloatSlider(value=3.0, description='k [N·m/rad]', max=20.0, min=0.5, step=0.5), FloatSli…

---
<a id='s5'></a>
## Section 5 — Control Theory Deep-Dive

### Layer 1: PD Joint Control (in MuJoCo actuators)
$$\tau_i = K_p(\theta^*_i - \theta_i) - K_d \dot{\theta}_i$$
This is a classical **proportional-derivative** feedback controller. MuJoCo's `position` actuator implements this automatically with gains `kp` and `kv`.

### Layer 2: Resolved-Rate (Task-Space) Control
Each timestep, we compute how far the tip is from the target and command joint velocities:
$$\dot{\theta} = J^\dagger(\theta) \cdot \underbrace{(p^* - p)}_\text{task-space error} / \Delta t$$
This is **kinematic feedback control** — the "controller" is the IK solver.

### Layer 3: Admittance Control (Force Regulation)
$$d_{\text{offset}} \leftarrow d_{\text{offset}} + K_f \cdot (F_n^* - F_n)$$
Force error adjusts the **position** target. This is the admittance paradigm (force error → position modification). It's the opposite of impedance control (position error → force output).

### Layer 4: Null-Space Control (Redundancy Resolution)
With 3 joints and 2 task DOF, there is one free dimension. We exploit it:
$$\dot{\theta} = J^\dagger \dot{p}^* + \underbrace{(I - J^\dagger J)}_{N} \dot{\theta}_0$$
$N$ is the **null-space projector** — anything multiplied by $N$ produces zero tip velocity but can still move the joints.

In [6]:
# ── Admittance control demo ─────────────────────────────────────────────────

def demo_admittance_control(kp=80, kd=8, kf=0.003, fn_target=3.75,
                             fn_disturbance=2.0, sim_time=5.0):
    """
    Simulate the admittance + PD control loop in 1D:
    pen presses against a spring board, admittance control regulates force.
    """
    dt    = 0.002
    steps = int(sim_time / dt)
    ts    = np.arange(steps) * dt

    # 1D simplified: board spring, arm position control
    board_k      = BOARD_K
    board_b      = BOARD_B
    board_I      = BOARD_INERTIA
    arm_stiffness = 200.0    # effective arm stiffness [N/m]

    # State variables
    phi         = 0.0;    phi_dot   = 0.0
    d_offset    = 0.004;  # initial penetration
    fn_history  = [];     phi_history = [];    d_history = []
    target_h    = []

    for i, t in enumerate(ts):
        # Simulated disturbance: sudden change in external torque at t=2s
        tau_ext = fn_disturbance if t > 2.0 else 0.0

        # Contact force (simplified: proportional to overlap)
        Fn = max(0.0, arm_stiffness * d_offset - tau_ext)

        # Board dynamics
        moment_arm = SWEEP_OFFSET
        tau_contact = Fn * moment_arm
        phi_ddot    = (tau_contact - board_k * phi - board_b * phi_dot) / board_I
        phi_dot    += phi_ddot * dt
        phi        += phi_dot  * dt

        # Admittance control: force error → offset update
        d_offset   += kf * (fn_target - Fn) * dt / 0.002  # normalize
        d_offset    = max(0.0, min(d_offset, 0.02))

        fn_history.append(Fn)
        phi_history.append(np.degrees(phi))
        d_history.append(d_offset * 1000)  # mm
        target_h.append(fn_target)

    # Plot
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    fig.suptitle('Admittance Control + Board Dynamics (1D Simplified)', fontsize=13)

    ax = axes[0]
    ax.set_ylabel('Fₙ [N]')
    ax.plot(ts, fn_history,  color='#ff9944', lw=2, label='Fₙ (actual)')
    ax.axhspan(F_MIN, F_MAX, alpha=0.15, color='#44ff99')
    ax.axhline(fn_target,    color='#44ff99', ls='--', lw=1.5, label=f'F* = {fn_target} N')
    ax.axvline(2.0, color='#ff6666', ls=':', lw=1.5, label='Disturbance at t=2s')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.set_ylabel('Board tilt φ [deg]')
    ax.plot(ts, phi_history, color='#66aaff', lw=2)

    ax = axes[2]
    ax.set_ylabel('Pen offset d [mm]')
    ax.set_xlabel('Time [s]')
    ax.plot(ts, d_history,   color='#cc88ff', lw=2)
    ax.axhline(0, color='#555577', ls=':', lw=1)

    plt.tight_layout()
    plt.show()

interact(
    demo_admittance_control,
    kf         = FloatSlider(value=0.003, min=0.0001, max=0.02, step=0.0001,
                             description='Kf (admit.)',  readout_format='.4f'),
    fn_target  = FloatSlider(value=F_TARGET, min=F_MIN, max=F_MAX, step=0.1, description='F* [N]'),
    fn_disturbance = FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.1, description='Disturbance [N·m]'),
);

interactive(children=(IntSlider(value=80, description='kp', max=240, min=-80), IntSlider(value=8, description=…

---
<a id='s6'></a>
## Section 6 — Full Simulation & Results Replay

### Run the simulation

In [7]:
# Run the full simulation (headless, saves data)
# ⚠️  This requires mujoco to be installed: pip install mujoco
# ⚠️  Takes ~30s to run

import subprocess, sys

print('Running simulation...')
result = subprocess.run(
    [sys.executable, 'simulate.py', '--no-video'],
    capture_output=True, text=True, cwd=os.getcwd()
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✅ Simulation done!')

Running simulation...
[MuJoCo] Loading: c:\Users\LENOVO\.gemini\antigravity\scratch\chopstick_crane_2\model.xml
[Sim]    Duration: 40.0s  (20000 steps, dt=0.002s)
[Controller] Home config: theta = [-76.5 106.6  -3.9] deg
  t=0.0s  phase=WARMUP      Fn=0.00N  phi=0.0 deg  [wall: 0.0s]
  t=1.0s  phase=WARMUP      Fn=0.00N  phi=-0.0 deg  [wall: 0.2s]
[t=2.00s] -> SWEEP phase
  t=2.0s  phase=SWEEP       Fn=0.00N  phi=0.0 deg  [wall: 0.5s]
  t=3.0s  phase=SWEEP       Fn=8.89N  phi=9.3 deg  [wall: 0.8s]
  t=4.0s  phase=SWEEP       Fn=4.06N  phi=10.5 deg  [wall: 1.1s]
  t=5.0s  phase=SWEEP       Fn=1.06N  phi=3.2 deg  [wall: 1.4s]
  t=6.0s  phase=SWEEP       Fn=2.36N  phi=2.5 deg  [wall: 1.6s]
  t=7.0s  phase=SWEEP       Fn=4.13N  phi=6.6 deg  [wall: 1.8s]
  t=8.0s  phase=SWEEP       Fn=22.79N  phi=6.1 deg  [wall: 2.0s]
  t=9.0s  phase=SWEEP       Fn=22.48N  phi=1.8 deg  [wall: 2.2s]
  t=10.0s  phase=SWEEP       Fn=1.53N  phi=10.0 deg  [wall: 2.4s]
  t=11.0s  phase=SWEEP       Fn=4.15N  phi=9

In [8]:
# Load and display results
import os

DATA_PATH = 'results/data.npz'

if not os.path.exists(DATA_PATH):
    print(f'❌ Data file not found: {DATA_PATH}')
    print('   Run the cell above first.')
else:
    logs   = dict(np.load(DATA_PATH, allow_pickle=True))
    t      = logs['t']
    tip    = logs['tip']
    target = logs['target']
    phi    = logs['phi']
    Fn     = logs['Fn']
    phase  = logs['phase']
    theta  = logs['theta']

    sweep  = (phase == 2)
    err    = np.linalg.norm(tip - target, axis=1)

    print(f'📊 Loaded {len(t)} timesteps  ({t[-1]:.2f}s total)')
    print(f'\n── Sweep Phase Statistics ──')
    if np.any(sweep):
        print(f'  Tracking error: mean={err[sweep].mean()*1000:.1f}mm  max={err[sweep].max()*1000:.1f}mm')
        print(f'  Contact force:  mean={Fn[sweep].mean():.2f}N  '
              f'in-band={100*np.mean((Fn[sweep]>=F_MIN)&(Fn[sweep]<=F_MAX)):.1f}%')
        print(f'  Board tilt:     range=[{np.degrees(phi[sweep]).min():.1f}°, '
              f'{np.degrees(phi[sweep]).max():.1f}°]')

📊 Loaded 19252 timesteps  (38.50s total)

── Sweep Phase Statistics ──
  Tracking error: mean=5.4mm  max=15.0mm
  Contact force:  mean=5.25N  in-band=67.4%
  Board tilt:     range=[0.0°, 10.8°]


In [9]:
# ── Interactive plot explorer ──────────────────────────────────────────────

if 'logs' in dir():
    from plot_results import plot_all
    plot_all(logs, out_dir='results')
else:
    print('Load logs first (run the cell above).')

[Plots]  Saved 5 figures to results/

-- Sweep Phase Statistics --
  Tracking error: mean=5.4mm  max=15.0mm
  Contact force:  mean=5.25N  in-band=67.4%
  Board tilt:     mean=6.6 deg  range=[0.0 deg, 10.8 deg]


In [10]:
# ── Animate the arm sweep ────────────────────────────────────────────────
# (runs without MuJoCo — pure kinematics replay)

if 'logs' in dir():
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML, display

    sweep_idx = np.where(phase == 2)[0]   # only sweep frames
    step      = max(1, len(sweep_idx) // 80)   # ~80 frames
    frame_idx = sweep_idx[::step]

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.set_xlim(-0.5, 1.0);  ax.set_ylim(-0.5, 0.5)
    ax.set_aspect('equal');  ax.set_title('Sinusoidal Sweep — Arm Animation')
    ax.set_xlabel('x [m]');  ax.set_ylabel('z [m]')

    # Board (static for animation; in reality it tilts)
    board_line, = ax.plot([], [], color='#aa8855', lw=6, solid_capstyle='round', alpha=0.7)
    target_line, = ax.plot([], [], color='#33dd88', lw=2, ls='--', label='Target', alpha=0.7)
    trace_line,  = ax.plot([], [], color='#ff6644', lw=1, alpha=0.5, label='Pen trace')

    arm_lines  = [ax.plot([], [], lw=5, solid_capstyle='round',
                          color=['#ff6688', '#66aaff', '#88ff99'][i])[0] for i in range(3)]
    tip_dot,   = ax.plot([], [], 'o', color='#ff3333', ms=8, zorder=6)
    hinge_dot, = ax.plot(*BOARD_HINGE_POS, '*', color='#ffdd44', ms=10, zorder=7)
    time_text  = ax.text(0.02, 0.95, '', transform=ax.transAxes, color='#c8c8d8', fontsize=9)

    ax.scatter(0, 0, color='white', s=80, marker='s')
    ax.legend(loc='upper right', fontsize=9)

    tip_trace_x, tip_trace_z = [], []

    def animate(i):
        idx  = frame_idx[i]
        th   = theta[idx]
        ph   = phi[idx]
        si   = logs['s'][idx]
        pts  = all_joint_positions(th)
        tip_pos = forward_kinematics(th)
        tip_trace_x.append(tip_pos[0]);  tip_trace_z.append(tip_pos[1])

        # Board (tilted)
        board_pts = np.array([board_to_world(-0.04, ph), board_to_world(0.30, ph)])
        board_line.set_data(board_pts[:, 0], board_pts[:, 1])

        # Target curve
        sv = np.linspace(0, 1, 50)
        tc = np.array([target_curve(s, ph) for s in sv])
        target_line.set_data(tc[:, 0], tc[:, 1])

        # Arm links
        for j in range(3):
            arm_lines[j].set_data([pts[j][0], pts[j+1][0]], [pts[j][1], pts[j+1][1]])
        tip_dot.set_data([tip_pos[0]], [tip_pos[1]])
        trace_line.set_data(tip_trace_x, tip_trace_z)
        time_text.set_text(f't={t[idx]:.2f}s  s={si:.2f}  φ={np.degrees(ph):.1f}°')
        return arm_lines + [tip_dot, board_line, target_line, trace_line, time_text]

    ani = FuncAnimation(fig, animate, frames=len(frame_idx),
                        interval=60, blit=True)
    plt.close()
    display(HTML(ani.to_jshtml()))
else:
    print('Load data first (run the simulation cells above).')